# Differential Allergen Analysis (Cancer vs. Healthy) Boxplots, bubble plot, heatmap, and volcano plot for the top differentially abundant allergen proteins.

## Imports & Settings

In [ ]:
# ---- standard library ----
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  

# ---- data & single-cell ----
import numpy as np
import pandas as pd
import scanpy as sc

# ---- plotting ----
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from adjustText import adjust_text

# ---- stats / clustering (heatmap) ----
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage

from pathlib import Path

# =========================================================
# PATHS
# =========================================================
# Input data lives under DATA_DIR (override with: export DATA_DIR=/path/to/data)
# Generated figures go under RESULTS_DIR, kept separate from the data folder
# so outputs aren't lost if data/ is gitignored.
DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))
RESULTS_DIR = Path(os.environ.get("RESULTS_DIR", "../results"))
FIGURES_DIR = RESULTS_DIR / "Figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams["font.family"] = "Times New Roman"

# =========================================================
# LOAD DATA
# =========================================================
adata_ranked_proteins_path = str(DATA_DIR / "H5AD" / "adata_ranked_proteins.h5ad")
adata_ranked_proteins = sc.read_h5ad(adata_ranked_proteins_path)
print(adata_ranked_proteins)

## Boxplots

In [ ]:
def plot_top_upregulated_allergen_boxplots(
    adata,
    group_key="label",
    group_pos="Cancer",
    group_neg="Healthy",
    allergen_key="byMajority",
    alpha=0.05,
    logfc_thresh=0.01,
    top_n=10
):
    """Boxplots of the top upregulated allergen proteins (Cancer vs Healthy)."""
    COLOR_HEALTHY = "#0B2545"
    COLOR_CANCER = "#DC143C"

    flierprops = dict(
        marker="o",
        markerfacecolor="none",
        markeredgecolor="black",
        markersize=5,
        linestyle="none"
    )

    # ---- differential-expression results ----
    res = adata.uns["rank_proteins_groups"]
    names = np.array(res["names"]["Cancer"])
    logfc = np.array(res["logfoldchanges"]["Cancer"])
    padj = np.clip(np.array(res["pvals_adj"]["Cancer"]), 1e-300, None)

    is_allergen = adata.var.loc[names, allergen_key].values == "Allergen"
    sel = (padj < alpha) & (logfc > logfc_thresh) & is_allergen

    idx = np.where(sel)[0]
    idx = idx[np.argsort(padj[idx])]  # most significant first
    top_idx = idx[:top_n]

    X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
    groups = adata.obs[group_key].values

    fig, axes = plt.subplots(
        2, 5,
        figsize=(20, 8),
        gridspec_kw={"wspace": 0.6, "hspace": 0.30}
    )
    axes = axes.flatten()

    def iqr_mask(vals):
        """Boolean mask for values within 1.5*IQR of the box (used to skip plotting outliers as scatter)."""
        q1 = np.percentile(vals, 25)
        q3 = np.percentile(vals, 75)
        iqr = q3 - q1
        low = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        return (vals >= low) & (vals <= high)

    for k, i in enumerate(top_idx):
        ax = axes[k]
        gene = names[i]
        j = adata.var_names.get_loc(gene)

        cancer_vals = X[groups == group_pos, j]
        healthy_vals = X[groups == group_neg, j]

        bp = ax.boxplot(
            [healthy_vals, cancer_vals],
            labels=[group_neg, group_pos],
            patch_artist=True,
            showfliers=True,
            flierprops=flierprops,
            widths=0.65,
            medianprops=dict(color="black", linewidth=1, linestyle="--")
        )
        bp["boxes"][0].set_facecolor(COLOR_HEALTHY)
        bp["boxes"][1].set_facecolor(COLOR_CANCER)
        for box in bp["boxes"]:
            box.set_alpha(0.75)

        # jittered scatter of in-range points only
        mask_h = iqr_mask(healthy_vals)
        mask_c = iqr_mask(cancer_vals)

        ax.scatter(
            np.random.normal(1, 0.04, mask_h.sum()),
            healthy_vals[mask_h],
            color=COLOR_HEALTHY, alpha=0.5, s=10
        )
        ax.scatter(
            np.random.normal(2, 0.04, mask_c.sum()),
            cancer_vals[mask_c],
            color=COLOR_CANCER, alpha=0.5, s=10
        )

        # significance stars + adjusted p-value
        p = padj[i]
        if p < 0.001:
            star = "***"
        elif p < 0.01:
            star = "**"
        elif p < 0.05:
            star = "*"
        else:
            star = "ns"
        p_label = f"{p:.3e}"

        # robust y-limits (based on IQR whiskers, not raw min/max)
        y_all = np.concatenate([healthy_vals, cancer_vals])
        q1, q3 = np.percentile(y_all, [25, 75])
        iqr = q3 - q1
        data_min = min(y_all.min(), q1 - 1.5 * iqr)
        data_max = max(y_all.max(), q3 + 1.5 * iqr)
        y_range = data_max - data_min + 1e-9

        y_bottom = data_min - max(0.10 * y_range, 0.05 * abs(data_min))
        y_top = data_max + max(0.40 * y_range, 0.10 * abs(data_max))
        ax.set_ylim(y_bottom, y_top)

        ax.text(
            1.5, y_top - 0.05 * (y_top - y_bottom),
            f"{star}\n(Padj={p_label})",
            ha="center", va="top", fontsize=10, fontweight="bold"
        )
        ax.set_title(f"{k+1}. {gene}", fontsize=10)

    for k in range(len(top_idx), len(axes)):
        axes[k].axis("off")

    legend_elements = [
        Line2D([], [], color="none", label="Groups:"),
        Patch(facecolor=COLOR_HEALTHY, label=group_neg),
        Patch(facecolor=COLOR_CANCER, label=group_pos)
    ]
    leg = fig.legend(handles=legend_elements, loc="lower center", ncol=3, frameon=False, fontsize=12)
    for text in leg.get_texts():
        if text.get_text() == "Groups:":
            text.set_fontweight("bold")

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    save_path = str(FIGURES_DIR / "plot_top_10_upregulated_allergens_in_cancer_boxplots.png")
    plt.savefig(save_path, dpi=1500, bbox_inches="tight")
    plt.show()


plot_top_upregulated_allergen_boxplots(adata_ranked_proteins)

## Bubble Plot

In [ ]:
def plot_allergen_bubble_impact(data, figsize=(12, 12), title=None, color_up="#F4A261", color_down="#2A9D8F"):
    df = pd.DataFrame(data)

    # clip extreme logFC, keep upregulated only, sort ascending for plotting
    df["logFC"] = df["logFC"].clip(-0.07, 0.07)
    df = df[df["logFC"] > 0].reset_index(drop=True)
    df = df.sort_values(by="logFC", ascending=True).reset_index(drop=True)

    x = df["logFC"].values
    y = np.arange(len(df))

    # normalize -log10(FDR) to a discrete 1-7 scale for bubble size
    fdr = df["-log10FDR"].values
    fdr_norm = (fdr - fdr.min()) / (fdr.max() - fdr.min() + 1e-9)
    fdr_scaled = np.round(1 + fdr_norm * 6).astype(int)

    size_map = {i: 40 + (i - 1) * 60 for i in range(1, 8)}
    size_scaled = np.array([size_map[v] for v in fdr_scaled])

    bubble_colors = np.where(x < 0, color_down, color_up)

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(x, y, s=size_scaled, c=bubble_colors, alpha=0.85, edgecolor="black", linewidth=0.4)

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xlim(-0.08, 0.08)
    ax.set_xticks([-0.06, -0.04, -0.02, 0.0, 0.02, 0.04, 0.06])
    ax.set_yticks(y)
    ax.set_yticklabels(df["feature"])
    ax.tick_params(axis="y", labelsize=8)
    ax.set_xlabel("Log2FC", fontsize=12)
    ax.set_ylabel("Allergens", fontsize=12)
    ax.tick_params(axis="x", labelsize=11, width=1)
    ax.set_title(title or "Top Upregulated Allergens", fontsize=14)
    ax.grid(axis="x", linestyle="--", alpha=0.3)

    # ---- unified legend (abundance direction + significance scale) ----
    handles, labels = [], []

    handles.append(Line2D([], [], linestyle="none")); labels.append("Abundance")
    handles.append(Line2D([0], [0], marker="o", color="w", markerfacecolor=color_up,
                           markeredgecolor="black", markersize=10))
    labels.append("Upregulated")
    handles.append(Line2D([], [], linestyle="none")); labels.append(" ")

    handles.append(Line2D([], [], linestyle="none")); labels.append(r"$-\log_{10}(p_{\mathrm{adj}})$")
    for v in range(1, 8):
        handles.append(Line2D([0], [0], marker="o", color="black", linestyle="none",
                               alpha=0.3, markersize=np.sqrt(size_map[v])))
        labels.append(str(v))

    leg = ax.legend(
        handles=handles, labels=labels, loc="center left", bbox_to_anchor=(1.02, 0.5),
        frameon=True, prop={"size": 10}, labelspacing=1.2, borderpad=0.8, handletextpad=0.8
    )
    leg.get_frame().set_edgecolor("black")
    leg.get_frame().set_linewidth(1)
    leg.get_frame().set_facecolor("white")

    section_titles = {"Abundance", r"$-\log_{10}(p_{\mathrm{adj}})$"}
    for text in leg.get_texts():
        if text.get_text().strip() in section_titles:
            if text.get_text().strip() != r"$-\log_{10}(p_{\mathrm{adj}})$":
                text.set_fontsize(13)
                text.set_fontweight("bold")
            else:
                text.set_fontsize(11.5)

    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.savefig(str(FIGURES_DIR / "plot_top_regulated_allergens_bubble.png"), dpi=1500, bbox_inches="tight")
    plt.show()

    return df


# top allergens by |logFC| from the differential-expression results (both directions)
data = [
    {"feature": "A0A239YF41_9FIRM", "logFC": -0.02955, "-log10FDR": 5.754},
    {"feature": "E6KWI6_9PAST", "logFC": -0.02811, "-log10FDR": 4.442},
    {"feature": "D6GSY8_FILAD", "logFC": -0.06503, "-log10FDR": 4.252},
    {"feature": "X8ISL9_9STRE", "logFC": -0.01777, "-log10FDR": 3.637},
    {"feature": "A0A2K9HAQ2_9BACT", "logFC": -0.01731, "-log10FDR": 3.518},
    {"feature": "A0A6L5DSP1_9BACT", "logFC": -0.02142, "-log10FDR": 3.199},
    {"feature": "UPI000469B5B7", "logFC": -0.01600, "-log10FDR": 3.047},
    {"feature": "A0A3N2MZC8_9BACT", "logFC": -0.03159, "-log10FDR": 2.915},
    {"feature": "A0A379EY57_9BACT", "logFC": -0.02975, "-log10FDR": 2.915},
    {"feature": "L1MCS5_9BACT", "logFC": -0.01899, "-log10FDR": 2.834},
    {"feature": "D1BP93_VEIPT", "logFC": 0.03986, "-log10FDR": 5.059},
    {"feature": "F3B2V5_9FIRM", "logFC": 0.03836, "-log10FDR": 2.862},
    {"feature": "A0A2W5KWT7_9CORY", "logFC": 0.02282, "-log10FDR": 2.726},
    {"feature": "A0A2S7ZLE7_9FIRM", "logFC": 0.03866, "-log10FDR": 2.666},
    {"feature": "W1VMM1_STRPA", "logFC": 0.01489, "-log10FDR": 2.272},
    {"feature": "D2ZSM3_NEIMU", "logFC": 0.02536, "-log10FDR": 2.083},
    {"feature": "A0A2T4TA75_9FIRM", "logFC": 0.02578, "-log10FDR": 2.014},
    {"feature": "H1HWB9_9FIRM", "logFC": 0.02092, "-log10FDR": 1.968},
    {"feature": "A0A1E9MPU5_9NEIS", "logFC": 0.03474, "-log10FDR": 1.961},
    {"feature": "K0XE54_9FIRM", "logFC": 0.02809, "-log10FDR": 1.948},
]

plot_allergen_bubble_impact(data)

## Heatmap

In [ ]:
def plot_top_upregulated_allergens_heatmap(
    adata,
    allergen_key="byMajority",
    allergen_value="Allergen",
    alpha=0.05,
    logfc_thresh=0.01,
    top_n=10,
    figsize=(12, 10),
    cmap="RdBu_r"
):
    """Clustered heatmap of the top upregulated allergen proteins (Cancer vs Healthy)."""

    res = adata.uns["rank_proteins_groups"]
    names = np.array(res["names"]["Cancer"])
    logfc = np.array(res["logfoldchanges"]["Cancer"])
    padj = np.clip(np.array(res["pvals_adj"]["Cancer"]), 1e-300, None)
    neglog = -np.log10(padj)

    if allergen_key not in adata.var.columns:
        raise ValueError(f"{allergen_key} not found in adata.var")

    is_allergen = adata.var.loc[names, allergen_key].astype(str).values == allergen_value
    sel = (padj < alpha) & (logfc > logfc_thresh) & is_allergen

    idx = np.where(sel)[0]
    print(f"Significant allergen proteins (Cancer-up): {len(idx)}")
    if len(idx) == 0:
        raise ValueError("No allergen proteins pass DE filters.")

    idx = idx[np.argsort(neglog[idx])[::-1]]  # strongest signal first
    if top_n is not None:
        idx = idx[:top_n]
    proteins = [names[i] for i in idx if names[i] in adata.var_names]

    # row-wise z-scored expression matrix (proteins x samples)
    X = adata[:, proteins].X
    if hasattr(X, "toarray"):
        X = X.toarray()
    X = StandardScaler().fit_transform(X).T
    df = pd.DataFrame(X, index=proteins, columns=adata.obs_names)

    # order samples by group so Healthy/Cancer sit side by side
    groups = adata.obs["label"]
    order = np.argsort(groups.values)
    df = df.iloc[:, order]
    groups = groups.iloc[order]

    palette = {"Cancer": "#DC143C", "Healthy": "#0B2545"}
    col_colors = groups.map(palette)

    row_linkage = linkage(df.values, method="ward")
    vlim = np.max(np.abs(df.values))

    g = sns.clustermap(
        df,
        row_linkage=row_linkage,
        col_cluster=False,
        cmap=cmap,
        vmin=-vlim,
        vmax=vlim,
        figsize=figsize,
        xticklabels=False,
        yticklabels=True,
        col_colors=col_colors,
        dendrogram_ratio=(0.15, 0.1),
        cbar_pos=(0.02, 0.8, 0.03, 0.18)
    )
    g.cax.set_title("Z-score", fontsize=12, pad=10)

    if top_n and top_n > 50:
        g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=4)

    # clean up dendrogram/colorbar strip
    g.ax_col_dendrogram.set_visible(False)
    g.ax_col_colors.set_yticks([])
    g.ax_col_colors.set_ylabel("")
    for spine in g.ax_col_colors.spines.values():
        spine.set_visible(False)

    # white separator line between the two sample groups
    order_groups = groups.values
    boundary = np.where(order_groups[:-1] != order_groups[1:])[0]
    for b in boundary:
        g.ax_heatmap.axvline(b + 0.5, color="white", linewidth=2)

    # group labels above the color strip
    ax = g.ax_col_colors
    ax.text(0.25, 1.4, "Healthy", ha="center", va="bottom", transform=ax.transAxes,
            color=palette["Healthy"], fontsize=15, fontweight="bold")
    ax.text(0.75, 1.4, "Cancer", ha="center", va="bottom", transform=ax.transAxes,
            color=palette["Cancer"], fontsize=15, fontweight="bold")

    plt.savefig(str(FIGURES_DIR / "plot_heatmap_top_upregulated_allergens_in_cancer.png"),
                dpi=1500, bbox_inches="tight")
    plt.show()

    return g


plot_top_upregulated_allergens_heatmap(adata_ranked_proteins, top_n=10)

## Volcano Plot 

In [ ]:
def plot_allergen_volcano(adata, alpha=0.05, logfc_thresh=0.01, figsize=(12, 8), top_n_labels=3):
    COLOR_UP = "#F25C54"
    COLOR_DOWN = "#00A896"
    COLOR_NEUTRAL = "#AAAAAA"

    result = adata.uns["rank_proteins_groups"]
    names = np.array(result["names"]["Cancer"])
    logfc = np.array(result["logfoldchanges"]["Cancer"])
    pvals_adj = np.clip(np.array(result["pvals_adj"]["Cancer"]), 1e-300, None)
    neg_log10_p = -np.log10(pvals_adj)

    sig_mask = neg_log10_p > -np.log10(alpha)
    up_mask = (logfc > logfc_thresh) & sig_mask
    down_mask = (logfc < -logfc_thresh) & sig_mask
    size = 20 + (neg_log10_p * 18)
    is_allergen = adata.var.loc[names, "byMajority"].values == "Allergen"

    fig, ax = plt.subplots(figsize=figsize)

    def scatter(mask, marker):
        ax.scatter(
            logfc[mask], neg_log10_p[mask],
            c=np.where(up_mask[mask], COLOR_UP, np.where(down_mask[mask], COLOR_DOWN, COLOR_NEUTRAL)),
            s=size[mask], alpha=0.75, marker=marker, edgecolor="none"
        )

    scatter(~is_allergen, "o")
    scatter(is_allergen, "*")

    # label the top allergens on each side
    texts = []
    left_mask = is_allergen & (logfc < -logfc_thresh) & sig_mask
    left_top = np.where(left_mask)[0][np.argsort(neg_log10_p[np.where(left_mask)[0]])[::-1]][:top_n_labels]

    right_mask = is_allergen & (logfc > logfc_thresh) & sig_mask
    right_top = np.where(right_mask)[0][np.argsort(neg_log10_p[np.where(right_mask)[0]])[::-1]][:top_n_labels]

    for i in list(left_top) + list(right_top):
        texts.append(ax.text(
            logfc[i], neg_log10_p[i], names[i], fontsize=9, weight="bold",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75)
        ))

    adjust_text(
        texts, ax=ax,
        force_text=(0.3, 0.4),
        force_static=(0.0, 0.0),
        force_pull=(0.05, 0.05),
        expand=(1.15, 1.25),
        max_move=(25, 25),
        arrowprops=dict(arrowstyle="-", lw=0.5, color="black", shrinkA=2)
    )

    ax.axhline(-np.log10(alpha), color="black", linestyle="--", linewidth=1)
    ax.text(ax.get_xlim()[1], -np.log10(alpha), f"  FDR = {alpha}", va="bottom", ha="left", fontsize=10)
    ax.axvline(logfc_thresh, color="black", linestyle="--", linewidth=1)
    ax.axvline(-logfc_thresh, color="black", linestyle="--", linewidth=1)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-0.07, 0.07)
    ax.set_xlabel(r"$\log_{2}(\mathrm{fold\ change})$", fontsize=13)
    ax.set_ylabel(r"$-\log_{10}(p_{\mathrm{adj}})$", fontsize=13)
    ax.set_title("Differential protein abundance (cancer vs healthy)")
    fig.subplots_adjust(right=0.72)

    # ---- unified legend ----
    handles, labels = [], []
    handles.append(Line2D([], [], linestyle="none")); labels.append("Abundance")
    handles += [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=COLOR_UP, markersize=10),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=COLOR_DOWN, markersize=10),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=COLOR_NEUTRAL, markersize=10)
    ]
    labels += ["Increased", "Decreased", "Not significant"]
    handles.append(Line2D([], [], linestyle="none")); labels.append(" ")

    handles.append(Line2D([], [], linestyle="none")); labels.append("Allergens")
    handles += [
        Line2D([0], [0], marker="*", color="w", markerfacecolor=COLOR_UP, markersize=15),
        Line2D([0], [0], marker="*", color="w", markerfacecolor=COLOR_DOWN, markersize=15),
        Line2D([0], [0], marker="*", color="w", markerfacecolor=COLOR_NEUTRAL, markersize=15),
    ]
    labels += ["Increased", "Decreased", "Not significant"]
    handles.append(Line2D([], [], linestyle="none")); labels.append(" ")

    handles.append(Line2D([], [], linestyle="none"))
    labels.append(r"$\boldsymbol{-\log_{10}(p_{\mathrm{adj}})}$")
    for v in range(1, 8):
        handles.append(Line2D([0], [0], marker="o", color="black", alpha=0.3, linestyle="none",
                               markersize=np.sqrt(40 + v * 80) / 2))
        labels.append(str(v))

    leg = ax.legend(handles=handles, labels=labels, loc="center left", bbox_to_anchor=(1.02, 0.5),
                     frameon=True, prop={"size": 10}, labelspacing=0.8, borderpad=0.8)
    leg.get_frame().set_edgecolor("black")
    leg.get_frame().set_linewidth(1)
    leg.get_frame().set_facecolor("white")

    section_titles = {"Abundance", r"$\boldsymbol{-\log_{10}(p_{\mathrm{adj}})}$", "Allergens"}
    for text in leg.get_texts():
        if text.get_text().strip() in section_titles:
            text.set_fontsize(13)
            text.set_fontweight("bold")

    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / "volcano_plot_labeled.png"), dpi=1500, bbox_inches="tight")
    plt.show()


plot_allergen_volcano(adata_ranked_proteins, top_n_labels=3)